## XBRL US API - Python example  
This sample Python code queries the XBRL US Public Filings Database.
### Authenticate for access token 
Run the cell below, then type your XBRL US Web account email, account password, Client ID, and secret (get these from https://xbrl.us/access-token), pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return any or all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
print('Enter your XBRL US Web account email: ')
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode
email = input()
password = getpass.getpass(prompt='Password: ')
clientid = getpass.getpass(prompt='Client ID: ')
secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : ''.join(email), 
            'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'password' : ''.join(password), 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

payload = urlencode(body_auth)
url = 'https://api.xbrl.us/oauth2/token'
headers = {"Content-Type": "application/x-www-form-urlencoded"}

res = requests.request("POST", url, data=payload, headers=headers)
auth_json = res.json()

if 'error' in auth_json:
    print ("\n\nThere was a problem generating an access token with these credentials. Run the first cell again to enter credentials.")
else:
    print ("\n\nYour access token expires in 60 minutes. After it expires, run the cell immediately below this one to generate a new token and continue to use the query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
access_token = auth_json['access_token']
refresh_token = auth_json['refresh_token']
newaccess = ''
newrefresh = ''
#print('access token: ' + access_token + ' refresh token: ' + refresh_token)

#### Refresh token 
The cell below is only needed to refresh an expired access token after 60 minutes. When the access token no longer returns results, run the cell below to refresh the access token or re-enter credentials by running the cell above. Until the refresh token process is needed, skip ahead to **Make a Query**. 


In [ ]:
token = token if newrefresh != '' else refresh_token 

refresh_auth = {'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'grant_type' : 'refresh_token', 
            'platform' : 'ipynb', 
            'refresh_token' : ''.join(token) }
refreshres = requests.post(url, data=refresh_auth)
refresh_json = refreshres.json()
access_token = refresh_json['access_token']
refresh_token = refresh_json['refresh_token']#print('access token: ' + access_token + 'refresh token: ' + refresh_token)
print('Your access token is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.')
print(access_token)

### Make a query 
After the access token confirmation appears above, you can modify the query below, then use the **_Cell >> Run_** menu option from the cell **immediately below this text** to run the entire query for results.

There are two searches in separate routines below - the first identifies dts.id values for Apple 10-Ks and the second uses the hard-coded values in a query to return any instance of the defined text string pattern in the entire document - this search is restricted to XBRL US Member accounts.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/ for other endpoints and parameters to filter and return. 

In [ ]:
# Define the parameters for the filter and fields to be returned, 
# run the loop to return results 
# Use any of these singly: 
# fact | dts | entity | network | relationship etc. 
# see https://xbrlus.github.io/xbrl-api for appropriate 
# search parameters and fields for each endpoint.

endpoint = 'report'
offset_value = 0
res_df = []

# Define the parameters of the query

sic_code = [3571
           ] 

reports = ['10-K',
           '10-K/A'
        ]

# OR search by company (comment out SIC code in param) 

companies_cik = [#'0000320193', ## Apple (AAPL)
                ]

# Define data fields to return (multi-sort based on order)

fields = ['report.filing-date.sort(DESC)',
          'report.entity-name',
          'report.document-type',
          'dts.id'
         ]

string_years = [str(int) for int in years]
string_sic = [str(int) for int in sic_code]

params = {'report.sic-code': ','.join(string_sic),
         'report.document-type': ','.join(reports),
         #'entity.cik': ','.join(companies_cik),
         'fields': ','.join(fields)
         }

# Execute the query with loop for all results

search_endpoint = 'https://api.xbrl.us/api/v1/'+endpoint+'/search'
orig_fields = params['fields']

count = 0
query_start = datetime.now()
printed = False
while True:
    if not printed:
        print("On", query_start.strftime("%c"), email, "(client ID:", str(clientid.split('-')[0]), "...) started the query and")
        printed = True
    res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
    res_json = res.json()
    if 'error' in res_json:
        print('There was an error: {}'.format(res_json['error_description']))
        break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(pd.DataFrame(res_df).to_html()))

On Mon Jan 17 13:55:13 2022 //email// (client ID: 9d3618e3 ...) started the query and
up to 2000 records are found so far ...
 - this set contained fewer than the 2000 possible, only 102 records.

At Mon Jan 17 13:55:13 2022, the query finished with   102   rows returned in -1 day, 23:59:59.938799 for 
https://api.xbrl.us/api/v1/report/search?report.sic-code=3571&report.document-type=10-K,10-K/A&fields=report.filing-date.sort(DESC),report.entity-name,report.document-type,dts.id


,report.filing-date,report.entity-name,report.document-type,dts.id
0,2021-10-29,Apple Inc.,10-K,505653
1,2021-08-27,"Super Micro Computer, Inc.",10-K,493972
2,2021-03-26,Dell Technologies Inc.,10-K,450469
3,2021-03-25,"One Stop Systems, Inc.",10-K,450164
4,2021-03-23,"Socket Mobile, Inc.",10-K,449583
5,2021-02-24,"OMNICELL, INC.",10-K,441926
6,2020-10-30,Apple Inc.,10-K,418604
7,2020-09-15,"CCUR Holdings, Inc.",10-K,412227
8,2020-08-31,"Super Micro Computer, Inc.",10-K,410430
9,2020-03-27,Dell Technologies Inc.,10-K,374761


In [ ]:
# Define the parameters for the filter and fields to be returned, 
# run the loop to return results 
# Use any of these singly: 
# fact | dts | entity | network | relationship etc. 
# see https://xbrlus.github.io/xbrl-api for appropriate 
# search parameters and fields for each endpoint.

endpoint = 'document' #Use any of these singly: fact | dts | entity | network | relationship etc. - see https://xbrlus.github.io/xbrl-api for appropriate search parameters and fields for each endpoint
offset_value = 0
res_df = []

# Define the parameters of the query 

text_search = ['21.1 NEAR/2 subsidiaries' 
                ]

# Define data fields to return (multi-sort based on order)

fields = [ 'document-type',
           'document.uri',
           'dts.id.sort(DESC)',
           'document.example'
         ]

params = { 'dts.id': '505653,493972,450469,450164,449583,441926,418604,412227,410430,374761,374483,374568,368136,367772,359056',
         'document.text-search': ','.join(text_search),
         'fields': ','.join(fields)
         }

# Execute the query with loop for all results

search_endpoint = 'https://api.xbrl.us/api/v1/'+endpoint+'/search'
orig_fields = params['fields']

count = 0
query_start = datetime.now()
printed = False
while True:
    if not printed:
        print("On", query_start.strftime("%c"), email, "(client ID:", str(clientid.split('-')[0]), "...) started the query and")
        printed = True
    res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
    res_json = res.json()
    if 'error' in res_json:
        print('There was an error: {}'.format(res_json['error_description']))
        break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(pd.DataFrame(res_df).to_html()))

On Mon Jan 17 14:01:59 2022 //email// (client ID: 9d3618e3 ...) started the query and
up to 2000 records are found so far ...
 - this set contained fewer than the 2000 possible, only 8 records.

At Mon Jan 17 14:02:04 2022, the query finished with   8   rows returned in 0:00:04.205868 for 
https://api.xbrl.us/api/v1/document/search?dts.id=505653,493972,450469,450164,449583,441926,418604,412227,410430,374761,374483,374568,368136,367772,359056&document.text-search=21.1+NEAR/2+subsidiaries&fields=document-type,document.uri,dts.id.sort(DESC),document.example


,document.uri,dts.id,document.example
0,http://www.sec.gov/Archives/edgar/data/320193/000032019320000096/a10-kexhibit2119262020.htm,418604,"EX-<b>21.1</b> 5 a10-kexhibit2119262020.htm EX-<b>21.1</b> Document Exhibit <b>21.1 Subsidiaries</b> of Apple Inc ... (b)(<b>21</b>)(ii) of Regulation S-K, the names of other <b>subsidiaries</b> ... would not constitute a significant <b>subsidiary</b> as of the end of ..."
1,http://www.sec.gov/Archives/edgar/data/320193/000032019320000096/aapl-20200926.htm,418604,"... it designs and develops <b>nearly</b> the entire solution for ... it designs and develops <b>nearly</b> the entire solution for ... <b>2</b>.32 $ <b>2</b>.09 Diluted $ 3.28 $ <b>2</b>.97 $ <b>2</b>.98 $ <b>2</b>.30 $ <b>2</b>.08 ... August 18, 2020. <b>21.1** Subsidiaries</b> of the Registrant. 23.<b>1</b>** Consent of ..."
2,http://www.sec.gov/Archives/edgar/data/749038/000110465920105213/tm2024746d1_ex21-1.htm,412227,"EX-<b>21.1</b> 3 tm2024746d1_ex21-<b>1</b>.htm EXHIBIT <b>21.1</b> Exhibit <b>21.1 Subsidiaries</b> of CCUR Holdings, Inc. ... Each of the below listed <b>subsidiaries</b> ... following the recapitalization of this <b>subsidiary</b>. All of the below ..."
3,http://www.sec.gov/Archives/edgar/data/1375365/000137536520000064/smci-ex2112020630x10k.htm,410430,"... -<b>21.1</b> 11 smci-ex2112020630x10k.htm EXHIBIT <b>21.1</b> Exhibit EX-<b>21.1</b> 3 exhibit211.htm <b>SUBSIDIARIES</b> ... OF SUPER MICRO COMPUTER, INC. EXHIBIT <b>21.1 SUBSIDIARIES</b> ..."
4,https://www.sec.gov/Archives/edgar/data/1394056/000156459020013196/oss-10k_20191231.htm,374568,"... <b>1</b>.8 <b>1</b>.8 <b>2</b>.6 3.<b>2</b> 18.92 % Other <b>2</b>.4 <b>2</b>.8 3.<b>2 2</b> ... with Customers, which superseded <b>nearly</b> all existing revenue recognition ... with Customers, which superseded <b>nearly</b> all existing revenue recognition ... as of July <b>1,</b> 2019. <b>21.1</b> List of <b>Subsidiaries</b>. 23.<b>1</b> Consent of ..."
5,https://www.sec.gov/Archives/edgar/data/926326/000092632620000010/omcl-20191231.htm,367772,"... headquarters located in Northern California, <b>near</b> major earthquake faults, and where ... costs was approximately $ <b>2.2</b> million, $ <b>2</b>.3 million, and $ <b>1</b>.6 million for ... 11/18/2019 <b>21.1+ Subsidiaries</b> of the Registrant 23.<b>1</b>+ Consent of Independent Registered ..."
6,http://www.sec.gov/Archives/edgar/data/926326/000092632620000010/omcl-20191231.htm,367772,"... headquarters located in Northern California, <b>near</b> major earthquake faults, and where ... costs was approximately $ <b>2.2</b> million, $ <b>2</b>.3 million, and $ <b>1</b>.6 million for ... 11/18/2019 <b>21.1+ Subsidiaries</b> of the Registrant 23.<b>1</b>+ Consent of Independent Registered ..."
7,https://www.sec.gov/Archives/edgar/data/1375365/000137536519000079/smci-ex211_2019630x10k.htm,359056,"... -<b>21.1</b> 9 smci-ex211_2019630x10k.htm EXHIBIT <b>21.1</b> Exhibit EX-<b>21.1</b> 3 exhibit211.htm <b>SUBSIDIARIES</b> ... OF SUPER MICRO COMPUTER, INC. EXHIBIT <b>21.1 SUBSIDIARIES</b> ..."


In [ ]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

df = pd.DataFrame(res_df)
df.to_csv(r"D:\results.csv",sep=",")